# Preparing the datasets for an LSTM-based Authorized Push Payment (APP) Fraud Detection Model
This notebook is used to prepare the datasets from IBM Synthetic Datasets, which are then used for training an LSTM-based Authorized Push Payment (APP) fraud detection model. The datasets are loaded from four different CSV files and then combined into one DataFrame. This enriched dataset provides the features needed to train a fraud detection model that can learn patterns from account characteristics, bank metrics, and transaction behaviors. 

The notebook saves the combined DataFrame into a single CSV file, to be used for training a preprocessing + LSTM model subsequently.


In [ ]:
import os
import pandas as pd

datasets_dir   = "./datasets/"
processed_datasets_dir = "./datasets_processed/"
os.makedirs(processed_datasets_dir, exist_ok=True)

processed_dataset_csv = "us_small_bank_xfers_processed.csv"

## Data Preparation
This code loads and enriches bank transfer transaction data for fraud detection by joining three datasets: banks, accounts (people + companies), and transactions. It creates a comprehensive dataset where each transaction is enriched with contextual features from both the sender's account (account type, currency, branch size, bank statistics) and recipient's account (country, entity type), while filtering out cash transactions and converting data types for efficient processing. The final merged dataset contains transaction details plus account-level and bank-level features that help the LSTM model identify fraudulent Authorized Push Payment (APP) patterns.

In [ ]:
print("Loading datasets...")

banks_cols     = ["Bank_ID", "Num_Transactions", "Num_Total_Locations"]
accts_cols     = ["Financial_Institution_ID", "Financial_Institution_Name", "Branch",
                  "Account_ID", "Account_Country", "Account_Currency", "Entity_Type",
                  "Account_Type", "Max_Overdraft"]
bank_xfers_cols = ["Transaction_Number", "Transaction_Date", "Transaction_Time",
                   "Transaction_Day_of_Week", "From_Bank", "From_Account", "To_Bank",
                   "To_Account", "Amount_Paid", "Payment_Currency", "Payment_Format", 
                   "From_Initial_Balance", "Transaction_Type", "Is_APP_Fraud",
                   "Sufficient_Funds", "Overdraft_Okay", "Is_Hold"]

banks_df           = pd.read_csv(datasets_dir + "us_small_banks.csv", usecols=banks_cols, dtype=str)
accts_people_df    = pd.read_csv(datasets_dir + "us_small_liquid_accts_people.csv", usecols=accts_cols, dtype=str)
accts_companies_df = pd.read_csv(datasets_dir + "us_small_liquid_accts_companies.csv",
                                 encoding="iso8859_2", usecols=accts_cols, dtype=str)
bank_xfers_df      = pd.read_csv(datasets_dir + "us_small_bank_xfers-chrono.csv", usecols=bank_xfers_cols, dtype=str)

print("Preparing datasets...")

accts_df = pd.concat([accts_people_df, accts_companies_df], ignore_index=True)

branches_df = (accts_df.groupby(["Financial_Institution_ID", "Branch"])["Account_ID"]
               .nunique().to_frame(name="Branch_Account_Count").reset_index())
branches_df = branches_df.merge(
    banks_df[["Bank_ID", "Num_Transactions", "Num_Total_Locations"]], 
    left_on="Financial_Institution_ID", right_on="Bank_ID", how="left"
    ).drop("Bank_ID", axis="columns"
    ).rename(columns={
        "Num_Transactions": "Bank_Num_Transactions", 
        "Num_Total_Locations": "Bank_Num_Total_Locations"})

accts_df = accts_df.merge(branches_df, on=["Financial_Institution_ID", "Branch"], how="left")

dataset_df = bank_xfers_df[(bank_xfers_df["From_Bank"] != "Cash") & (bank_xfers_df["To_Bank"] != "Cash")]

dataset_df["Timestamp"] = (dataset_df["Transaction_Date"] + " " + 
                           dataset_df["Transaction_Time"])

from_cols = ["Financial_Institution_Name", "Account_ID", "Account_Country", "Account_Currency",
             "Entity_Type", "Account_Type", "Max_Overdraft",
             "Branch_Account_Count", "Bank_Num_Transactions", "Bank_Num_Total_Locations"]
dataset_df = dataset_df.merge(
    accts_df[from_cols],
    left_on=["From_Bank", "From_Account"],
    right_on=["Financial_Institution_Name", "Account_ID"], how="left"
    ).drop(["Financial_Institution_Name", "Account_ID"], axis="columns"
    ).rename(columns={
        "Account_Country": "From_Account_Country", 
        "Account_Currency": "From_Account_Currency",
        "Entity_Type": "From_Entity_Type", 
        "Account_Type": "From_Account_Type",
        "Max_Overdraft": "From_Account_Max_Overdraft",
        "Branch_Account_Count": "From_Branch_Account_Count",
        "Bank_Num_Transactions": "From_Bank_Num_Transactions",
        "Bank_Num_Total_Locations": "From_Bank_Num_Total_Locations"})

to_cols = ["Financial_Institution_Name", "Account_ID", "Account_Country", "Entity_Type"]
dataset_df = dataset_df.merge(
    accts_df[to_cols],
    left_on=["To_Bank", "To_Account"],
    right_on=["Financial_Institution_Name", "Account_ID"], how="left"
    ).drop(["Financial_Institution_Name", "Account_ID"], axis="columns"
    ).rename(columns={
        "Account_Country": "To_Account_Country",
        "Entity_Type": "To_Entity_Type"})

dataset_df = dataset_df.reset_index(drop=True)

# Sort based on key columns From_Bank, From_Account and Timestamp 
# to group together rows of same sender accounts in chronological order.
dataset_df = dataset_df.sort_values(["From_Bank", "From_Account", "Timestamp"]).reset_index(drop=True)

print("Dataset preparation complete!")
print(f"Dataset shape: {dataset_df.shape}  |  Fraud cases: {len(dataset_df[(dataset_df['Is_APP_Fraud'] == '1')])}")
dataset_df.info()



## Save to CSV
Save the prepared dataset to a CSV file.

In [ ]:
filepath = processed_datasets_dir + processed_dataset_csv
dataset_df.to_csv(filepath, index=False)
print("Prepared dataset saved to " + filepath)
